# FlowCycle 改进版：增加 Target 监督损失

原版 FlowCycle 只用循环一致性约束（L_src + α·L_align）来优化噪声，没有直接利用目标图像的监督信号。  
改进：增加第三项损失 L_target = MSE(去噪结果, ground truth target)，直接拉近编辑结果与期望目标。

**损失函数：**
```
loss = L_src + α·L_align + β·L_target
```
- L_src: 循环回来应该≈源图像
- L_align: 两个中间状态应该对齐  
- L_target (新增): 去噪结果应该≈目标图像  
- β = 2（大于 α，强调目标监督）

## Step 1: 检查 GPU & 安装依赖

In [ ]:
!nvidia-smi
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"bfloat16 支持: {torch.cuda.is_bf16_supported()}")

In [ ]:
!pip install -q diffusers==0.30.1 transformers==4.44.2 accelerate sentencepiece==0.2.0

## Step 2: 登录 HuggingFace

In [ ]:
from huggingface_hub import login
login()

## Step 3: 上传数据

上传数据，每组包含 `source.png`、`target.png` 和 `prompt.txt`（第一行源提示词，第二行目标提示词）。

In [ ]:
import os
from google.colab import files
from IPython.display import display
from PIL import Image

for t in data_dirs:
    os.makedirs(t, exist_ok=True)
    print(f"=== 请上传 {t}/source.png ===")
    for name, data in files.upload().items():
        with open(f"{t}/source.png", "wb") as f:
            f.write(data)
    print(f"=== 请上传 {t}/target.png ===")
    for name, data in files.upload().items():
        with open(f"{t}/target.png", "wb") as f:
            f.write(data)
    print(f"=== 请上传 {t}/prompt.txt ===")
    for name, data in files.upload().items():
        with open(f"{t}/prompt.txt", "wb") as f:
            f.write(data)

# 预览上传的数据
import matplotlib.pyplot as plt
n = len(data_dirs)
fig, axes = plt.subplots(2, n, figsize=(5 * n, 10))
if n == 1:
    axes = axes.reshape(2, 1)
for i, t in enumerate(data_dirs):
    axes[0, i].imshow(Image.open(f"{t}/source.png"))
    axes[0, i].set_title(f"{t} - Source")
    axes[0, i].axis("off")
    axes[1, i].imshow(Image.open(f"{t}/target.png"))
    axes[1, i].set_title(f"{t} - Target (GT)")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## Step 4: 定义工具函数

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from diffusers import StableDiffusion3Pipeline, FlowMatchEulerDiscreteScheduler, AutoencoderKL
from PIL import Image
from typing import Union, Tuple, Optional
from torchvision import transforms as tvt
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
import random
import numpy as np
import os
import matplotlib.pyplot as plt

# ======================== 工具函数 ========================

@torch.no_grad()
def img_to_latents(x, vae):
    x = 2. * x.to(device=vae.device, dtype=vae.dtype) - 1.
    posterior = vae.encode(x).latent_dist
    return posterior.mean * vae.config.scaling_factor

class Noises(nn.Module):
    def __init__(self, dtype, img_size):
        super().__init__()
        self.noises = nn.Parameter(torch.randn(2, 16, img_size // 8, img_size // 8, dtype=dtype))
    def forward(self, i):
        return self.noises[i].unsqueeze(0)

def load_image(imgname, target_size=None):
    pil_img = Image.open(imgname).convert('RGB')
    if target_size is not None:
        if isinstance(target_size, int):
            target_size = (target_size, target_size)
        pil_img = pil_img.resize(target_size, Image.Resampling.LANCZOS)
    return tvt.ToTensor()(pil_img)[None, ...]

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@torch.no_grad()
def get_text_embeddings(prompt_a, prompt_b, src_gs, tar_gs, pipe):
    pe, npe, ppe, nppe = pipe.encode_prompt(
        prompt=prompt_a, prompt_2=prompt_a, prompt_3=prompt_a,
        negative_prompt="", negative_prompt_2="", negative_prompt_3="",
        device=pipe.device, num_images_per_prompt=1,
        do_classifier_free_guidance=src_gs > 1.0)
    if src_gs > 1.0:
        ce_a = torch.cat([npe, pe], dim=0)
        pp_a = torch.cat([nppe, ppe], dim=0)
    else:
        ce_a, pp_a = pe, ppe

    pe, npe, ppe, nppe = pipe.encode_prompt(
        prompt=prompt_b, prompt_2=prompt_b, prompt_3=prompt_b,
        negative_prompt="", negative_prompt_2="", negative_prompt_3="",
        device=pipe.device, num_images_per_prompt=1,
        do_classifier_free_guidance=tar_gs > 1.0)
    if tar_gs > 1.0:
        ce_b = torch.cat([npe, pe], dim=0)
        pp_b = torch.cat([nppe, ppe], dim=0)
    else:
        ce_b, pp_b = pe, ppe
    return ce_a, pp_a, ce_b, pp_b

@torch.no_grad()
def get_pooled_embeddings(prompt, pipe):
    _, _, ppe, _ = pipe.encode_prompt(
        prompt=prompt, prompt_2=prompt, prompt_3=prompt,
        negative_prompt="", negative_prompt_2="", negative_prompt_3="",
        device=pipe.device, num_images_per_prompt=1,
        do_classifier_free_guidance=False)
    return ppe

def decode_latent_to_pil(latent, vae):
    with torch.no_grad():
        img = vae.decode(latent / vae.config.scaling_factor, return_dict=False)[0]
        img = (img / 2 + 0.5).clamp(0, 1)
        img = img[0].permute(1, 2, 0).float().cpu().numpy()
        return Image.fromarray((img * 255).round().astype("uint8"))

def denoise_loop(latents, timesteps, pipe, cond_embeds, pooled_proj, gs, device):
    for _, t in enumerate(timesteps):
        t_idx = pipe.scheduler.timesteps.tolist().index(t.item())
        sigma_t = pipe.scheduler.sigmas[t_idx]
        sigma_next = pipe.scheduler.sigmas[t_idx + 1]
        dt = sigma_t - sigma_next
        inp = latents.repeat(2, 1, 1, 1) if gs > 1.0 else latents
        t_batch = t.expand(inp.shape[0]).to(device)
        with torch.inference_mode():
            pred = pipe.transformer(
                hidden_states=inp, timestep=t_batch,
                encoder_hidden_states=cond_embeds,
                pooled_projections=pooled_proj,
                return_dict=True).sample
        if gs > 1.0:
            u, c = pred.chunk(2)
            pred = u + gs * (c - u)
        latents = latents - dt * pred
    return latents


print("所有函数定义完成！")

## Step 5: 加载 SD3 模型 & 设置超参数

In [ ]:
# 超参数
model_id = "stabilityai/stable-diffusion-3-medium-diffusers"
seed = 2025
beta = 0.2
num_inference_steps = 50
n_max = 33
source_guidance_scale = 3.5
target_guidance_scale = 5.5
lr = 0.1
epochs = 40
alpha = 0.2
img_size = 512
device = "cuda:0"
warmup_epochs = int(epochs * 0.1)
data_dirs = ["tuple1"]

# 自动选择精度
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"使用精度: {dtype}")

set_seed(seed)

# 加载模型
print("加载 SD3 模型...")
pipe = StableDiffusion3Pipeline.from_pretrained(model_id, torch_dtype=dtype).to(device)
pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(num_inference_steps, device=device)
timesteps = pipe.scheduler.timesteps[num_inference_steps - n_max:]
sigma_max_t = pipe.scheduler.sigmas[pipe.scheduler.timesteps.tolist().index(timesteps[0])]
print("模型加载完成！")

## Phase 1: 训练可学习噪声（带 Target 监督）

使用原版 FlowCycle 的直接噪声优化架构（`Noises` 类），但增加第三项损失：  
- L_src: 循环一致性（Source→Target→Source ≈ Source）  
- L_align: 中间状态对齐  
- **L_target (新增)**: MSE(去噪得到的 target, ground truth target latent)  

`loss = L_src + α·L_align + β·L_target`，β=0.02 强调弱目标监督。

In [ ]:
# ---- 预处理：编码所有数据（含 target） ----
all_data = []
for data_dir in data_dirs:
    with open(os.path.join(data_dir, "prompt.txt"), 'r') as f:
        lines = f.read().strip().split('\n')
    source_prompt, target_prompt = lines[0].strip(), lines[1].strip()

    # 编码 source 图像
    src_img = load_image(os.path.join(data_dir, "source.png"), img_size).to(device=device, dtype=dtype)
    a = img_to_latents(src_img, pipe.vae)

    # 编码 target 图像（ground truth）
    tar_img = load_image(os.path.join(data_dir, "target.png"), img_size).to(device=device, dtype=dtype)
    gt_tar_latent = img_to_latents(tar_img, pipe.vae)

    ce_a, pp_a, ce_b, pp_b = get_text_embeddings(
        source_prompt, target_prompt, source_guidance_scale, target_guidance_scale, pipe)

    all_data.append({
        'name': data_dir,
        'img_latent': a,
        'gt_tar_latent': gt_tar_latent,
        'ce_a': ce_a, 'pp_a': pp_a,
        'ce_b': ce_b, 'pp_b': pp_b,
    })
    print(f"已编码 {data_dir}: src={source_prompt[:40]}... tar={target_prompt[:40]}...")

# ---- 为每组数据初始化可学习噪声 ----
all_noises = []
all_optimizers = []
all_schedulers = []

for sample in all_data:
    noises = Noises(dtype, img_size).to(device)
    optimizer = optim.Adam(noises.parameters(), lr=lr)
    warmup = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs, eta_min=1e-5)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])
    all_noises.append(noises)
    all_optimizers.append(optimizer)
    all_schedulers.append(scheduler)

print(f"\n初始化 {len(all_data)} 组可学习噪声")

# ---- 训练 ----
losses_history = []
print(f"开始训练, epochs={epochs}, alpha={alpha}, beta={beta}\n")

for epoch in range(epochs):
    epoch_loss = 0.0

    for i, sample in enumerate(all_data):
        all_optimizers[i].zero_grad()
        a = sample['img_latent']

        eps_src = all_noises[i](0)  # 可学习噪声 source
        eps_tar = all_noises[i](1)  # 可学习噪声 target

        # Source → Target 去噪
        latents = a * (1 - sigma_max_t) + sigma_max_t * eps_src
        tmp_a = latents
        denoised_tar = denoise_loop(latents, timesteps, pipe, sample['ce_b'], sample['pp_b'],
                                    target_guidance_scale, device)

        # Target → Source 去噪（循环回来）
        latents = denoised_tar * (1 - sigma_max_t) + sigma_max_t * eps_tar
        tmp_b = latents
        recovered_src = denoise_loop(latents, timesteps, pipe, sample['ce_a'], sample['pp_a'],
                                     source_guidance_scale, device)

        # 三项损失
        loss1 = F.mse_loss(a, recovered_src, reduction='none').mean(dim=(1, 2, 3))          # L_src
        loss2 = F.mse_loss(tmp_a, tmp_b, reduction='none').mean(dim=(1, 2, 3))               # L_align
        loss3 = F.mse_loss(denoised_tar, sample['gt_tar_latent'], reduction='none').mean(dim=(1, 2, 3))  # L_target
        loss = loss1 + alpha * loss2 + beta * loss3

        loss.backward()
        all_optimizers[i].step()
        epoch_loss += loss.item()

    for s in all_schedulers:
        s.step()

    avg_loss = epoch_loss / len(all_data)
    losses_history.append(avg_loss)

    if epoch % 10 == 0:
        print(f"  epoch {epoch}/{epochs}, loss={avg_loss:.6f} (L_src={loss1.item():.4f}, L_align={loss2.item():.4f}, L_target={loss3.item():.4f}), lr={all_optimizers[0].param_groups[0]['lr']:.2e}")

print("\n训练完成！")

# 保存噪声
for i, sample in enumerate(all_data):
    torch.save(all_noises[i].state_dict(), os.path.join(sample['name'], "learned_noises.pt"))
    print(f"已保存 {sample['name']}/learned_noises.pt")

# 画训练曲线
plt.figure(figsize=(8, 4))
plt.plot(losses_history)
plt.xlabel("Epoch")
plt.ylabel("Loss (L_src + α·L_align + β·L_target)")
plt.title("Training Loss")
plt.yscale("log")
plt.grid(True)
plt.show()

## Phase 2: 用训练好的噪声推理 + 可视化

用优化后的可学习噪声做 Source→Target 去噪，得到编辑结果。

## 对比：随机噪声 vs 改进版（+L_target）

用纯随机高斯噪声（无训练）做同样的去噪，和我们训练后的结果、ground truth 一起对比。

In [ ]:
for i, sample in enumerate(all_data):
    data_dir = sample['name']
    print(f"[{i+1}/{len(all_data)}] {data_dir}: 推理中...")

    with torch.no_grad():
        eps_src = all_noises[i](0)
        latents = sample['img_latent'] * (1 - sigma_max_t) + sigma_max_t * eps_src
        latents = denoise_loop(latents, timesteps, pipe, sample['ce_b'], sample['pp_b'],
                               target_guidance_scale, device)
        result = decode_latent_to_pil(latents, pipe.vae)
        result.save(os.path.join(data_dir, "edited_improved.png"))

    src_img = Image.open(os.path.join(data_dir, "source.png")).convert('RGB').resize((img_size, img_size))
    tar_img = Image.open(os.path.join(data_dir, "target.png")).convert('RGB').resize((img_size, img_size))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(src_img)
    axes[0].set_title(f"Source ({data_dir})")
    axes[0].axis("off")
    axes[1].imshow(result)
    axes[1].set_title("Edited (Ours: +L_target)")
    axes[1].axis("off")
    axes[2].imshow(tar_img)
    axes[2].set_title("Ground Truth Target")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()

print("推理完成！")